# Colab Runner: PPO + Imitation Learning

Runs the full pipeline on Colab. Code is cloned from GitHub; trained models and results are persisted to Google Drive so disconnects do not lose work.

**Compute note:** PPO training is CPU-bound (MuJoCo physics), so a GPU runtime does not speed up the expert; it only helps from-scratch BC. Use a GPU runtime for the BC ablation/sweep, CPU is fine for PPO. Long runs need Colab Pro or an active tab to avoid idle disconnects.

## 1. Clone the code from GitHub

In [ ]:
REPO = 'https://github.com/em-ech/rl-ppo-imitation-learning.git'
CODE_DIR = '/content/GroupProject'
import os, sys, shutil
os.chdir('/content')  # never stand inside the dir we are about to delete
if os.path.exists(CODE_DIR):
    shutil.rmtree(CODE_DIR)
!git clone -q {REPO} {CODE_DIR}
assert os.path.isdir(CODE_DIR), 'clone failed'
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print('code in', CODE_DIR, '->', sorted(os.listdir(CODE_DIR))[:8])

## 2. Mount Drive for persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/rl_project'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

## 3. Install pinned dependencies

Takes a few minutes. Restart the runtime only if Colab prompts about a numpy/torch version change.

In [ ]:
!pip -q install -r /content/GroupProject/requirements.txt

## 4. Configure persistence + sanity check

Pointing `PROJECT_DATA_ROOT` at Drive sends models/, data/, outputs/, logs/ to Drive so they survive disconnects. `os.environ` changes are inherited by the `!python ...` calls below.

In [ ]:
import sys, os  # re-assert in case the runtime was restarted
sys.path.insert(0, '/content/GroupProject'); os.chdir('/content/GroupProject')
os.environ['PROJECT_DATA_ROOT'] = DRIVE_ROOT
from src import config
print('device:', config.device())
print('models ->', config.MODELS_DIR)
import gymnasium as gym
for e in ['Walker2d-v4', 'Ant-v4']:
    gym.make(e).reset(seed=0); print('ok', e)

## 5. Stage 1 - PPO experts (M1)

Walker2d with the validated config; Ant with VecNormalize + a larger budget. Each writes best_model (and vecnormalize.pkl for Ant) to Drive. The `cd` prefix makes each cell independent of the working directory.

In [ ]:
!cd /content/GroupProject && python train_expert.py Walker2d-v4 4000000 4

In [ ]:
!cd /content/GroupProject && python train_expert.py Ant-v4 8000000 8 norm

## 6. Stage 2 - demonstration collection + EDA + quality gate (M2)

In [ ]:
!cd /content/GroupProject && python collect_demos.py Walker2d-v4 100
!cd /content/GroupProject && python collect_demos.py Ant-v4 100

## 7. Stage 3 - BC experiments (M3, M4, M5) and multi-seed arch sweep (M8)

Use a GPU runtime here for the from-scratch BC runs.

In [ ]:
!cd /content/GroupProject && python bc_experiments.py Walker2d-v4
!cd /content/GroupProject && python arch_sweep.py Walker2d-v4
# repeat for Ant once its expert and dataset are ready:
# !cd /content/GroupProject && python bc_experiments.py Ant-v4
# !cd /content/GroupProject && python arch_sweep.py Ant-v4

## 8. Stages 4-5 - DAgger and pretraining

Run `notebooks/04_dagger.ipynb` and `notebooks/05_pretraining.ipynb` (still in development). All outputs already persist to Drive via `PROJECT_DATA_ROOT`.

## Recovering after a disconnect

Models, checkpoints (every 100k steps), datasets, and figures live under `MyDrive/rl_project/`. Re-run cells 1-4 to remount and reattach, then continue. Note: `train_expert.py` restarts a run from scratch rather than resuming mid-training, so prefer Colab Pro or keep the tab active for the long expert runs.